## 1) Setup Env

In [1]:
from pathlib import Path

ROOT = Path.cwd()
for folder in [
    ROOT / "data" / "raw",
    ROOT / "data" / "processed",
    ROOT / "models",
    ROOT / "src",
    ROOT / "notebooks",
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Root project:", ROOT)
print("Struktur folder siap.")

Root project: d:\TugasUnud\Semester 6\BIG DATA\Analisis Sentimen\notebooks
Struktur folder siap.


In [2]:
import os
import time
from typing import List, Dict, Optional

import pandas as pd
from dotenv import load_dotenv
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

load_dotenv()

YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY", "").strip()
video_ids_raw = os.getenv("YOUTUBE_VIDEO_IDS", "")
YOUTUBE_VIDEO_IDS = [v.strip() for v in video_ids_raw.split(",") if v.strip()]

max_comments_raw = os.getenv("MAX_COMMENTS", "").strip()
MAX_COMMENTS = int(max_comments_raw) if max_comments_raw else None

print("API key loaded:", bool(YOUTUBE_API_KEY))
print("Jumlah video ID:", len(YOUTUBE_VIDEO_IDS))
print("Max comments per video:", "SEMUA" if MAX_COMMENTS is None else MAX_COMMENTS)

API key loaded: True
Jumlah video ID: 5
Max comments per video: SEMUA


In [ ]:
import json


def fetch_replies(
    youtube,
    video_id: str,
    parent_comment_id: str,
    retry: int = 5,
    raw_reply_pages: Optional[List[Dict]] = None,
) -> List[Dict[str, str]]:
    replies: List[Dict[str, str]] = []
    next_page_token = None

    while True:
        res = None
        for attempt in range(retry):
            try:
                req = youtube.comments().list(
                    part="snippet",
                    parentId=parent_comment_id,
                    maxResults=100,
                    pageToken=next_page_token,
                    textFormat="plainText",
                )
                res = req.execute()

                if raw_reply_pages is not None:
                    raw_reply_pages.append(
                        {
                            "endpoint": "comments.list",
                            "video_id": video_id,
                            "parent_comment_id": parent_comment_id,
                            "page_token": next_page_token,
                            "response": res,
                        }
                    )
                break
            except HttpError:
                if attempt == retry - 1:
                    return replies
                time.sleep(min(2 ** (attempt + 1), 30))

        if res is None:
            break

        items = res.get("items", [])
        for item in items:
            sn = item.get("snippet", {})
            replies.append(
                {
                    "video_id": video_id,
                    "comment_id": item.get("id", ""),
                    "parent_comment_id": parent_comment_id,
                    "is_reply": True,
                    "author": sn.get("authorDisplayName", ""),
                    "text": sn.get("textDisplay", ""),
                    "like_count": sn.get("likeCount", 0),
                    "published_at": sn.get("publishedAt", ""),
                }
            )

        next_page_token = res.get("nextPageToken")
        if not next_page_token:
            break

        time.sleep(0.2)

    return replies


def fetch_comments(
    api_key: str,
    video_id: str,
    max_comments: Optional[int] = None,
    retry: int = 5,
    include_replies: bool = True,
    raw_pages: Optional[List[Dict]] = None,
) -> List[Dict[str, str]]:
    if not api_key or not video_id:
        raise ValueError("YOUTUBE_API_KEY atau video_id tidak valid")

    youtube = build("youtube", "v3", developerKey=api_key)
    comments: List[Dict[str, str]] = []
    next_page_token = None
    page_count = 0
    seen_comment_ids = set()

    while True:
        res = None
        for attempt in range(retry):
            try:
                req = youtube.commentThreads().list(
                    part="snippet",
                    videoId=video_id,
                    maxResults=100,
                    pageToken=next_page_token,
                    order="time",
                    textFormat="plainText",
                )
                res = req.execute()

                if raw_pages is not None:
                    raw_pages.append(
                        {
                            "endpoint": "commentThreads.list",
                            "video_id": video_id,
                            "page_token": next_page_token,
                            "response": res,
                        }
                    )
                break
            except HttpError as err:
                status = getattr(err.resp, "status", None)
                if status == 403:
                    print(f"  ERROR 403 pada video {video_id}. Cek quota/API access.")
                    return comments
                if attempt == retry - 1:
                    print(f"  Gagal request thread page {page_count + 1} pada video {video_id}.")
                    return comments
                time.sleep(min(2 ** (attempt + 1), 30))

        if res is None:
            break

        items = res.get("items", [])
        if not items:
            break

        page_count += 1

        for item in items:
            top_obj = item.get("snippet", {}).get("topLevelComment", {})
            top_sn = top_obj.get("snippet", {})
            top_id = top_obj.get("id", "")

            if top_id and top_id not in seen_comment_ids:
                comments.append(
                    {
                        "video_id": video_id,
                        "comment_id": top_id,
                        "parent_comment_id": "",
                        "is_reply": False,
                        "author": top_sn.get("authorDisplayName", ""),
                        "text": top_sn.get("textDisplay", ""),
                        "like_count": top_sn.get("likeCount", 0),
                        "published_at": top_sn.get("publishedAt", ""),
                    }
                )
                seen_comment_ids.add(top_id)

            if include_replies and top_id:
                total_reply_count = item.get("snippet", {}).get("totalReplyCount", 0)
                if total_reply_count > 0:
                    replies = fetch_replies(
                        youtube,
                        video_id,
                        top_id,
                        retry=retry,
                        raw_reply_pages=raw_pages,
                    )
                    for rp in replies:
                        rp_id = rp.get("comment_id", "")
                        if rp_id and rp_id not in seen_comment_ids:
                            comments.append(rp)
                            seen_comment_ids.add(rp_id)

            if max_comments is not None and len(comments) >= max_comments:
                break

        if max_comments is not None and len(comments) >= max_comments:
            print(f"  Reached max_comments: {len(comments)}")
            break

        next_page_token = res.get("nextPageToken")
        if not next_page_token:
            break

        time.sleep(0.2)

    print(f"  Video {video_id}: selesai, page thread={page_count}, total komentar+balasan={len(comments)}")
    return comments


if not YOUTUBE_VIDEO_IDS:
    raise ValueError("Isi YOUTUBE_VIDEO_IDS di .env. Bisa banyak ID dipisahkan koma.")

all_comments = []
all_raw_pages: List[Dict] = []
video_metadata = []

print("=" * 60)
print("MULAI MENGAMBIL SEMUA DATA YOUTUBE (VIDEO + KOMENTAR + REPLIES)")
print("=" * 60)

youtube = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)

print("\n[STEP 1] Mengambil metadata video...")
for vid in YOUTUBE_VIDEO_IDS:
    try:
        req = youtube.videos().list(
            part="snippet,statistics,contentDetails,status",
            id=vid,
        )
        res = req.execute()
        
        all_raw_pages.append({
            "endpoint": "videos.list",
            "video_id": vid,
            "response": res,
        })
        
        if res.get("items"):
            video_info = res["items"][0]
            video_metadata.append({
                "video_id": vid,
                "title": video_info.get("snippet", {}).get("title", ""),
                "description": video_info.get("snippet", {}).get("description", ""),
                "channel_id": video_info.get("snippet", {}).get("channelId", ""),
                "channel_title": video_info.get("snippet", {}).get("channelTitle", ""),
                "published_at": video_info.get("snippet", {}).get("publishedAt", ""),
                "duration": video_info.get("contentDetails", {}).get("duration", ""),
                "view_count": video_info.get("statistics", {}).get("viewCount", 0),
                "like_count": video_info.get("statistics", {}).get("likeCount", 0),
                "comment_count": video_info.get("statistics", {}).get("commentCount", 0),
                "full_response": video_info,
            })
            print(f"  ✓ {vid}: {video_info.get('snippet', {}).get('title', 'Unknown')[:50]}")
        time.sleep(0.3)
    except Exception as e:
        print(f"  ✗ Error fetching video {vid}: {e}")

print("\n[STEP 2] Mengambil komentar dari semua video...")
for idx, vid in enumerate(YOUTUBE_VIDEO_IDS, 1):
    print(f"\n[{idx}/{len(YOUTUBE_VIDEO_IDS)}] Ambil video: {vid}")
    vid_comments = fetch_comments(
        YOUTUBE_API_KEY,
        vid,
        MAX_COMMENTS,
        retry=5,
        include_replies=True,
        raw_pages=all_raw_pages,
    )
    all_comments.extend(vid_comments)
    print(f"           Terambil: {len(vid_comments)}")

df_raw = pd.DataFrame(all_comments)
print(f"\n✓ Total comments: {len(df_raw)}")

df_video_metadata = pd.DataFrame(video_metadata)
video_metadata_path = ROOT / "data" / "raw" / "video_metadata.csv"
df_video_metadata.to_csv(video_metadata_path, index=False, encoding="utf-8")
print(f"✓ Video metadata tersimpan: {video_metadata_path}")

raw_path_csv = ROOT / "data" / "raw" / "comments_raw_all_videos.csv"
raw_path_parquet = ROOT / "data" / "raw" / "comments_raw_all_videos.parquet"
raw_path_json = ROOT / "data" / "raw" / "comments_raw_all_videos.json"

df_raw.to_csv(raw_path_csv, index=False, encoding="utf-8")
df_raw.to_parquet(raw_path_parquet, index=False)

with open(raw_path_json, "w", encoding="utf-8") as f:
    json.dump(all_raw_pages, f, ensure_ascii=False, indent=2, default=str)

print(f"\n✓ All raw API responses tersimpan: {raw_path_json}")
print(f"  Total API calls: {len(all_raw_pages)}")

from pymongo import MongoClient, UpdateOne
from pymongo.errors import BulkWriteError

MONGO_URI = os.getenv("MONGO_URI", "").strip()
MONGO_DB = os.getenv("MONGO_DB", "analisis_sentimen").strip()
MONGO_COMMENTS_COLLECTION = os.getenv("MONGO_COMMENTS_COLLECTION", "comments").strip()
MONGO_VIDEOS_COLLECTION = os.getenv("MONGO_VIDEOS_COLLECTION", "videos").strip()
MONGO_RAW_COLLECTION = os.getenv("MONGO_RAW_COLLECTION", "raw_api_responses").strip()

if MONGO_URI:
    print("\n[MONGODB] Menyimpan data...")
    try:
        client = MongoClient(MONGO_URI)
        db = client[MONGO_DB]
        
        print(f"  [1/3] Saving {len(df_raw)} comments to '{MONGO_COMMENTS_COLLECTION}'...")
        col_comments = db[MONGO_COMMENTS_COLLECTION]
        try:
            col_comments.create_index("comment_id", unique=True, sparse=True)
        except:
            pass
        
        records = df_raw.replace({pd.NA: None}).fillna("").to_dict(orient="records")
        if records:
            ops = []
            for r in records:
                key = {"comment_id": r.get("comment_id")}
                ops.append(UpdateOne(key, {"$setOnInsert": r}, upsert=True))
            
            if ops:
                try:
                    res = col_comments.bulk_write(ops, ordered=False)
                    inserted = getattr(res, "upserted_count", 0)
                    print(f"      ✓ Upserted {inserted} comments")
                except BulkWriteError as bwe:
                    print(f"      ✗ Bulk write error: {bwe.details}")
                except Exception as e:
                    print(f"      ✗ Error: {e}")
        
        print(f"  [2/3] Saving {len(video_metadata)} video metadata to '{MONGO_VIDEOS_COLLECTION}'...")
        col_videos = db[MONGO_VIDEOS_COLLECTION]
        try:
            col_videos.create_index("video_id", unique=True, sparse=True)
        except:
            pass
        
        video_records = [v.copy() for v in video_metadata]
        if video_records:
            ops = []
            for v in video_records:
                full_resp = v.pop("full_response", None)
                key = {"video_id": v.get("video_id")}
                v["full_response"] = full_resp
                ops.append(UpdateOne(key, {"$setOnInsert": v}, upsert=True))
            
            if ops:
                try:
                    res = col_videos.bulk_write(ops, ordered=False)
                    inserted = getattr(res, "upserted_count", 0)
                    print(f"      ✓ Upserted {inserted} videos")
                except Exception as e:
                    print(f"      ✗ Error: {e}")
        
        print(f"  [3/3] Saving {len(all_raw_pages)} raw API responses to '{MONGO_RAW_COLLECTION}'...")
        col_raw = db[MONGO_RAW_COLLECTION]
        try:
            col_raw.create_index([("endpoint", 1), ("video_id", 1), ("timestamp", 1)])
        except:
            pass
        
        import datetime
        raw_records = []
        for i, page in enumerate(all_raw_pages):
            raw_records.append({
                "_id": f"{page.get('endpoint')}_{page.get('video_id')}_{i}",
                "endpoint": page.get("endpoint"),
                "video_id": page.get("video_id"),
                "parent_comment_id": page.get("parent_comment_id"),
                "page_token": page.get("page_token"),
                "response": page.get("response"),
                "saved_at": datetime.datetime.utcnow(),
            })
        
        if raw_records:
            try:
                col_raw.insert_many(raw_records, ordered=False)
                print(f"      ✓ Inserted {len(raw_records)} raw API responses")
            except Exception as e:
                print(f"      ✗ Error: {e}")
        
        print("  ✓ MongoDB save completed!")
        
    except Exception as e:
        print(f"  ✗ MongoDB connection error: {e}")
else:
    print("  ✗ MONGO_URI not configured; skipping MongoDB save")

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Total comments:              {len(df_raw)}")
print(f"Total videos:                {len(video_metadata)}")
print(f"Total API calls recorded:    {len(all_raw_pages)}")
print(f"\nLocal files saved:")
print(f"  - Comments CSV:            {raw_path_csv}")
print(f"  - Comments Parquet:        {raw_path_parquet}")
print(f"  - Video metadata CSV:      {video_metadata_path}")
print(f"  - All raw API JSON:        {raw_path_json}")
print("=" * 60)

print("\nSample comments data:")
df_raw.head()


MULAI MENGAMBIL SEMUA DATA YOUTUBE (VIDEO + KOMENTAR + REPLIES)

[STEP 1] Mengambil metadata video...
  ✓ F6fgLwUeeqI: BATALKAN REVISI UU TNI
  ✓ MxCqHoldj2Y: RUU TNI Resmi Jadi UU! Ada yang Perlu Dikhawatirka
  ✓ sg8Mzx0fZbU: Kenapa UU TNI Banyak Ditolak? Ada Contoh Negara-ne
  ✓ 7CLZkPwhEG4: Revisi UU TNI: Apa dampaknya untuk masyarakat sipi
  ✓ KjXe214MfwQ: RUU TNI itu apa? Mirip Orde Baru? #shorts

[STEP 2] Mengambil komentar dari semua video...

[1/5] Ambil video: F6fgLwUeeqI
  Video F6fgLwUeeqI: selesai, page thread=35, total komentar+balasan=4327
           Terambil: 4327

[2/5] Ambil video: MxCqHoldj2Y
  Video MxCqHoldj2Y: selesai, page thread=10, total komentar+balasan=1255
           Terambil: 1255

[3/5] Ambil video: sg8Mzx0fZbU
  Video sg8Mzx0fZbU: selesai, page thread=23, total komentar+balasan=3689
           Terambil: 3689

[4/5] Ambil video: 7CLZkPwhEG4
  Video 7CLZkPwhEG4: selesai, page thread=33, total komentar+balasan=4718
           Terambil: 4718

[5/5] Ambil video